# Intelligence-Gated Settlement with Claude and DPX

Demonstrates the core DPX revenue model: an agent **pays for intelligence** before deciding whether to settle. Every micropayment (\$0.15–\$0.75 USDC on Base) buys a signal that Claude uses to reason about the settlement.

## What this shows

- **x402 micropayments** — the HTTP 402 payment protocol for machine-to-machine payments
- **Intelligence endpoints** — 23 live signals (macro stress, cascade risk, sovereign debt, FX corridors, climate, shipping, etc.)
- **AI-gated settlement** — Claude reasons across paid signals before authorizing payment
- **Audit trail** — every intelligence purchase and settlement is logged

## Architecture

```
Claude (claude-sonnet-5)
  │
  ├─ buy_intelligence(endpoint)  →  402 payment req  →  CDP facilitator  →  signal
  │     $0.15–$0.75 USDC per call, paid on Base mainnet
  │
  ├─ check_oracle_conditions     →  GET stability.untitledfinancial.com/reliability
  ├─ get_fee_quote               →  GET stability.untitledfinancial.com/quote
  ├─ screen_counterparty         →  GET agent.untitledfinancial.com/flow-check
  └─ execute_settlement          →  POST agent.untitledfinancial.com/settle
```

**Sandbox mode:** The intelligence endpoints require real x402 payment. The settlement step is sandboxed — no real funds move in the payment execution.

In [ ]:
%pip install anthropic httpx python-dotenv --quiet

In [ ]:
import os, json, httpx, anthropic
from dotenv import load_dotenv
load_dotenv()

client  = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
SANDBOX = os.environ.get('SANDBOX', 'true').lower() != 'false'
ORACLE  = 'https://stability.untitledfinancial.com'
AGENT   = 'https://agent.untitledfinancial.com'
INTEL   = 'https://intelligence.untitledfinancial.com'

# x402: agent wallet private key for signing intelligence payments
# Leave blank to use mock mode (signals simulated, no USDC spent)
AGENT_PRIVATE_KEY = os.environ.get('AGENT_PRIVATE_KEY', '')
MOCK_INTEL = not AGENT_PRIVATE_KEY

print(f'Sandbox: {SANDBOX}  |  Intelligence: {"mock" if MOCK_INTEL else "live x402"}')

## Intelligence endpoint catalogue

All 23 endpoints — each returns a structured signal that Claude can reason about.

In [ ]:
# Fetch live endpoint catalogue — no payment required
catalogue = httpx.get(f'{INTEL}/').json()
print(f"Intelligence API — {len(catalogue['endpoints'])} endpoints\n")
for ep in catalogue['endpoints']:
    print(f"  {ep['price']:6s}  {ep['path'].replace('/v1/intelligence/', ''):25s}  {ep.get('label', '')}")

## x402 payment flow

When an agent calls an intelligence endpoint without payment, it receives a `402 Payment Required` response with USDC payment details. The agent signs a transfer, retries, and receives the signal.

```
1. GET /v1/intelligence/macro-stress
   → 402: { accepts: [{ amount: '150000', asset: USDC, payTo: '0x...' }] }

2. Agent signs EIP-3009 transfer via CDP facilitator
   → X-Payment: <signed-token>

3. GET /v1/intelligence/macro-stress  (with X-Payment header)
   → 200: { regime: 'ELEVATED_RISK', score: 72, ... }
```

In mock mode below, step 2 is skipped and a simulated signal is returned.

In [ ]:
# ── Mock intelligence signals (used when no AGENT_PRIVATE_KEY is set) ──────
MOCK_SIGNALS = {
    'macro-stress': {
        'regime': 'ELEVATED_RISK', 'score': 72,
        'drivers': ['HY spread widening', 'TED spread 38bps', 'VIX 24'],
        'recommendation': 'Proceed with caution — monitor credit spreads'
    },
    'sovereign-debt': {
        'riskTier': 'MODERATE', 'score': 65,
        'flags': ['US deficit trajectory elevated', 'EM debt rollover stress Q3'],
        'recommendation': 'USD corridor stable; avoid EM exposure >30d'
    },
    'fx-settlement': {
        'corridor': 'USD/USD', 'stability': 'OPTIMAL',
        'executionRisk': 'LOW', 'volatility24h': '0.02%',
        'recommendation': 'Domestic USD corridor — proceed immediately'
    },
    'shipping-stress': {
        'globalStress': 'MODERATE', 'score': 58,
        'hotspots': ['Red Sea rerouting adding 8-12d', 'LA port congestion 3d'],
        'invoiceDelayRisk': 'LOW for domestic USD payments'
    },
}


def buy_intelligence(endpoint: str) -> dict:
    """Buy an intelligence signal via x402. Falls back to mock if no private key."""
    if MOCK_INTEL:
        name = endpoint.split('/')[-1]
        signal = MOCK_SIGNALS.get(name, {'status': 'mock', 'note': f'Mock signal for {name}'})
        print(f'  [mock] {endpoint} → {list(signal.keys())[:3]}...')
        return signal

    # ── Live x402 flow ───────────────────────────────────────────────────────
    url = f'{INTEL}{endpoint}'

    # Step 1: probe for payment requirements
    r = httpx.get(url, timeout=10)
    if r.status_code == 200:
        return r.json()  # already paid (cached)

    if r.status_code != 402:
        return {'error': f'Unexpected status {r.status_code}'}

    payment_req = r.json()
    accept      = payment_req['accepts'][0]
    amount      = int(accept['amount'])   # USDC atomic units (6 decimals)
    pay_to      = accept['payTo']
    asset       = accept['asset']
    print(f'  [x402] {endpoint} — ${amount/1e6:.4f} USDC → {pay_to[:10]}...')

    # Step 2: sign EIP-3009 transfer via CDP facilitator
    # Full implementation: https://github.com/coinbase/x402
    # For production, use the `x402` Python package or CDP SDK
    from eth_account import Account
    from eth_account.messages import encode_structured_data
    import time

    account = Account.from_key(AGENT_PRIVATE_KEY)
    valid_after  = int(time.time()) - 10
    valid_before = int(time.time()) + 300
    nonce        = os.urandom(32).hex()

    # EIP-3009 TransferWithAuthorization
    structured = {
        'types': {
            'EIP712Domain': [
                {'name': 'name',              'type': 'string'},
                {'name': 'version',           'type': 'string'},
                {'name': 'chainId',           'type': 'uint256'},
                {'name': 'verifyingContract', 'type': 'address'},
            ],
            'TransferWithAuthorization': [
                {'name': 'from',         'type': 'address'},
                {'name': 'to',           'type': 'address'},
                {'name': 'value',        'type': 'uint256'},
                {'name': 'validAfter',   'type': 'uint256'},
                {'name': 'validBefore',  'type': 'uint256'},
                {'name': 'nonce',        'type': 'bytes32'},
            ],
        },
        'primaryType': 'TransferWithAuthorization',
        'domain': {
            'name': 'USD Coin', 'version': '2',
            'chainId': 8453, 'verifyingContract': asset,
        },
        'message': {
            'from': account.address, 'to': pay_to,
            'value': amount, 'validAfter': valid_after,
            'validBefore': valid_before,
            'nonce': bytes.fromhex(nonce),
        },
    }
    signed = account.sign_message(encode_structured_data(structured))

    x_payment = json.dumps({
        'x402Version': 2, 'scheme': 'exact', 'network': 'eip155:8453',
        'payload': {
            'signature': signed.signature.hex(),
            'from': account.address, 'to': pay_to,
            'value': str(amount), 'validAfter': str(valid_after),
            'validBefore': str(valid_before), 'nonce': '0x' + nonce,
        },
    })

    # Step 3: retry with payment
    r2 = httpx.get(url, headers={'X-Payment': x_payment}, timeout=15)
    if r2.status_code == 200:
        return r2.json()
    return {'error': f'Payment failed: {r2.status_code}', 'body': r2.text[:200]}


print('buy_intelligence() ready')

## Settlement tools (free — no payment required)

In [ ]:
def check_oracle_conditions() -> dict:
    r = httpx.get(f'{ORACLE}/reliability', timeout=10).json()
    return {
        'status':    r.get('stability', {}).get('latestStatus', 'UNKNOWN'),
        'score':     r.get('stability', {}).get('currentScore', 0),
        'reasoning': r.get('intelligence', {}).get('reasoning', ''),
    }

def get_fee_quote(amount_usd, has_fx=False, esg_score=75) -> dict:
    r = httpx.get(f'{ORACLE}/quote',
        params={'amountUsd': amount_usd, 'hasFx': str(has_fx).lower(), 'esgScore': esg_score},
        timeout=10).json()
    return {
        'quote_id':  r.get('quoteId'),
        'total_bps': r.get('fees', {}).get('total', {}).get('bps'),
        'fee_usd':   r.get('fees', {}).get('total', {}).get('usd'),
        'net_usd':   r.get('settlement', {}).get('netUsd'),
    }

def screen_counterparty(amount, recipient_address) -> dict:
    r = httpx.get(f'{AGENT}/flow-check',
        params={'amount': amount, 'from': 'USD', 'to': 'USD', 'recipientAddress': recipient_address},
        timeout=15).json()
    return {
        'decision': r.get('decision', 'BLOCKED'),
        'reason':   r.get('holdReason') or r.get('blockReason') or 'All checks passed',
    }

def execute_settlement(amount, recipient_address, quote_id, purpose) -> dict:
    r = httpx.post(f'{AGENT}/settle', json={
        'amount': amount, 'sourceCurrency': 'USD', 'destinationCurrency': 'USD',
        'recipientAddress': recipient_address, 'purpose': purpose,
        'quoteId': quote_id, 'sandbox': SANDBOX,
    }, timeout=30).json()
    return {
        'status':        r.get('status'),
        'settlement_id': r.get('settlementId'),
        'tx_hash':       r.get('txHash'),
        'ai_decision':   r.get('aiDecision'),
        'ai_confidence': r.get('aiConfidence'),
    }

print('Settlement tools ready')

## Tool schemas for Claude

In [ ]:
tools = [
  {
    'name': 'buy_intelligence',
    'description': 'Buy a DPX intelligence signal via x402 micropayment ($0.15–$0.75 USDC on Base). Call this to get structured macro, climate, FX, sovereign, or cascade data before deciding whether to settle. Each call returns a fresh signal the agent can reason about.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'endpoint': {
          'type': 'string',
          'description': 'Intelligence endpoint path',
          'enum': [
            '/v1/intelligence/macro-stress',
            '/v1/intelligence/sovereign-debt',
            '/v1/intelligence/fx-settlement',
            '/v1/intelligence/cascade',
            '/v1/intelligence/shipping-stress',
            '/v1/intelligence/climate',
            '/v1/intelligence/supply-chain',
            '/v1/intelligence/currency-stress',
            '/v1/intelligence/mycelium',
            '/v1/intelligence/resonance',
          ]
        },
        'reason': {'type': 'string', 'description': 'Why this signal is relevant to the payment decision'}
      },
      'required': ['endpoint', 'reason']
    }
  },
  {
    'name': 'check_oracle_conditions',
    'description': 'Check global macro stability. Returns STABLE/CAUTION/UNSTABLE with score. Always call first — abort if UNSTABLE.',
    'input_schema': {'type': 'object', 'properties': {}, 'required': []}
  },
  {
    'name': 'get_fee_quote',
    'description': 'Get a binding fee quote (valid 300s). Returns quote_id, fee breakdown, and net settlement amount.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount_usd': {'type': 'number'},
        'has_fx':     {'type': 'boolean'},
        'esg_score':  {'type': 'number'}
      },
      'required': ['amount_usd', 'has_fx']
    }
  },
  {
    'name': 'screen_counterparty',
    'description': 'AML + sanctions + FATF R16 screen. Returns PROCEED, HOLD, or BLOCKED.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount':            {'type': 'number'},
        'recipient_address': {'type': 'string'}
      },
      'required': ['amount', 'recipient_address']
    }
  },
  {
    'name': 'execute_settlement',
    'description': 'Execute the settlement. Only call after intelligence review, oracle gate, quote, and PROCEED compliance screen.',
    'input_schema': {
      'type': 'object',
      'properties': {
        'amount':            {'type': 'number'},
        'recipient_address': {'type': 'string'},
        'quote_id':          {'type': 'string'},
        'purpose':           {'type': 'string'}
      },
      'required': ['amount', 'recipient_address', 'quote_id', 'purpose']
    }
  },
]

print(f'{len(tools)} tools defined')

## Intelligence-gated agent loop

In [ ]:
def dispatch_tool(name, inputs):
    if name == 'buy_intelligence':
        return json.dumps(buy_intelligence(inputs['endpoint']))
    if name == 'check_oracle_conditions':
        return json.dumps(check_oracle_conditions())
    if name == 'get_fee_quote':
        return json.dumps(get_fee_quote(**inputs))
    if name == 'screen_counterparty':
        return json.dumps(screen_counterparty(**inputs))
    if name == 'execute_settlement':
        return json.dumps(execute_settlement(**inputs))
    return json.dumps({'error': f'unknown tool {name}'})


def run_agent(task):
    print(f'\nTask: {task}\n' + '='*60)
    messages = [{'role': 'user', 'content': task}]
    system = """\
You are an intelligent payment agent with access to paid intelligence signals.
Before executing any large settlement, use buy_intelligence to purchase relevant signals.
Choose signals that are material to the payment — macro stress for large amounts,
FX corridor for cross-border, sovereign debt for EM counterparties, etc.
You are spending the agent's USDC to buy these signals — only buy what's relevant.

After reviewing intelligence, run the standard settlement flow:
1. check_oracle_conditions (abort if UNSTABLE)
2. get_fee_quote
3. screen_counterparty (abort if BLOCKED)
4. execute_settlement

Explain your reasoning at each step, including why you chose each intelligence signal
and how it influenced your settlement decision."""

    while True:
        resp = client.messages.create(
            model='claude-sonnet-5', max_tokens=4096,
            system=system, tools=tools, messages=messages,
        )
        for b in resp.content:
            if b.type == 'text' and b.text.strip():
                print(f'\nClaude: {b.text}')
        if resp.stop_reason == 'end_turn':
            return next((b.text for b in resp.content if b.type == 'text'), '')
        calls = [b for b in resp.content if b.type == 'tool_use']
        if not calls:
            break
        results = []
        for c in calls:
            print(f'\n  → {c.name}({json.dumps(c.input)[:80]}...)')
            out = dispatch_tool(c.name, c.input)
            print(f'  ← {out[:120]}...')
            results.append({'type': 'tool_result', 'tool_use_id': c.id, 'content': out})
        messages += [{'role': 'assistant', 'content': resp.content}, {'role': 'user', 'content': results}]
    return ''


print('Agent ready')

## Run it — $500K vendor settlement with intelligence gating

In [ ]:
result = run_agent(
    'Pay $500,000 USD to 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 '
    'for Global Logistics GmbH — Q2 procurement invoice #INV-2026-0088. '
    'This is a large payment. Buy whatever intelligence signals are relevant '
    'before deciding whether to proceed. Sandbox mode.'
)

## Run it — cross-border EM payment with sovereign risk check

In [ ]:
result = run_agent(
    'Pay $250,000 USD to 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 '
    'for a counterparty in an emerging market. '
    'Check sovereign debt risk and FX corridor conditions first. Sandbox mode.'
)

## Understanding x402 — the economics

Each intelligence call is a separate USDC micropayment on Base mainnet. The agent decides which signals are worth buying based on the payment context.

| Signal | Price | When to buy |
|---|---|---|
| `macro-stress` | \$0.15 | Any large settlement — credit regime classification |
| `fx-settlement` | \$0.25 | Cross-border payments — corridor stability + execution risk |
| `sovereign-debt` | \$0.25 | EM counterparties — sovereign risk tier + rollover stress |
| `shipping-stress` | \$0.25 | Payments tied to physical goods — freight + invoice delay risk |
| `supply-chain` | \$0.25 | Tech/manufacturing vendors — semiconductor + logistics risk |
| `cascade` | \$0.75 | Large or strategic payments — full cascade simulation |
| `mycelium` | \$0.50 | Systemic risk check — network topology + crisis detection |
| `resonance` | \$0.50 | When macro signals look correlated — phase alignment detection |

The agent bought relevant signals (~\$0.40–\$1.50 USDC total) and used them to decide whether $500K should move. That's the DPX intelligence revenue model.

## Next steps

- [Intelligence API reference](https://docs.untitledfinancial.com/api/intelligence-api) — all 23 endpoints with response schemas
- [x402 protocol guide](https://docs.untitledfinancial.com/integrations/x402) — full payment flow, EIP-3009, CDP facilitator
- [MCP server](https://docs.untitledfinancial.com/integrations/mcp) — 78 tools including `get_intelligence` for Claude Desktop
- **Go live**: set `AGENT_PRIVATE_KEY` in `.env` with a funded Base wallet to pay real x402 fees